# Phase 4 Mark: Hyperparameter Tuning + Error Analysis
**Date:** 2026-04-23 | **Researcher:** Mark Rodrigues | **Session:** 4 of 7

## Research Question
The Phase 3 champion (CLIP B/32 + category-conditioned + color rerank α=0.4) hits **R@1=0.6826**.
31.7% of queries still fail. Today we ask:
1. **Can systematic hyperparameter tuning push R@1 above 0.70?**
2. **What is the remaining failure mode?** Are failures recoverable (close misses) or fundamental (model blind spots)?
3. **Does higher color resolution (96D vs 48D) help?** More bins = better?  
4. **Does multiplicative fusion (soft-AND) beat additive?**

## Building on Anthony's Work
Anthony's Phase 3 found text metadata (R@1=60.2%) beats CLIP visual (55.3%), and that
CLIP L/14 with spatial+color+text hits R@1=0.6748. My Phase 3 found that **architecture
(category-conditioned retrieval) beats feature engineering** — zero new features gave +8.9pp.

Today I tune the architecture-driven Phase 3 champion and diagnose its failure modes.
Combined insight: if close misses dominate (correct product in top-5), then Phase 5
should add Anthony's text metadata as a **reranking signal** on top of my category filter.

## References
1. **Babenko et al., 2014** — "Neural Codes for Image Retrieval" — argues compact descriptors with appropriate pooling outperform fine-grained high-dimensional ones. Informs why 48D > 96D color.
2. **Johnson et al., 2019 (FAISS)** — "Billion-scale similarity search with GPUs" — per-category index partitioning reduces search space and improves precision.
3. **Liu et al., 2016** — "DeepFashion" dataset paper — notes that intra-class variation (lighting, viewpoint, pose) is the primary challenge in fashion retrieval. This is why lighting-robust features matter.

In [ ]:
import json, numpy as np, pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

RES = Path('../results')

# Load Phase 4 results
p4 = json.load(open(RES / 'phase4_mark_results.json'))['phase4_mark']
print('Phase 4 Mark results loaded')
print(f"Research question: {p4['research_question']}")
print(f"\nHeadline finding: {p4['headline_finding']}")

## Experiment 4.M.1: Champion Baseline Re-Validation
**Hypothesis:** Phase 3 champion (CLIP B/32 + cat.filter + color48 α=0.4) reproduces at R@1=0.6826.

In [ ]:
champ = p4['experiments']['4M1_champion_reval']
print('4.M.1 Champion re-validation:')
for k, v in champ.items():
    print(f'  {k}: {v:.4f}')
print(f"\nPhase 3 reported: R@1=0.6826 | Reproduced: R@1={champ['R@1']:.4f}")
print(f"Reproduction delta: {champ['R@1'] - 0.6826:+.4f} (should be ~0)")

## Experiment 4.M.2: Per-Category Alpha Optimization
**Hypothesis:** Different product categories benefit from different CLIP-vs-color blend ratios.
Sweaters (distinctive silhouette) → more CLIP. Tees (color is the differentiator) → more color.

**Method:** Grid search alpha ∈ {0.00, 0.05, ..., 1.00} independently per category.
Oracle upper bound: apply the best alpha per category to all queries.

**Note on overfitting:** Tuning alpha on the eval set is an oracle upper bound.
In production, tune on a held-out validation split. This shows the maximum extractable gain.

In [ ]:
percat = p4['experiments']['4M2_percat_alpha_oracle']
alpha_dict = p4['experiments']['4M2_best_alpha_per_cat']
alpha_scan = p4['alpha_scan_per_category']

print('Per-category optimal alphas:')
print(f"  {'Category':<16} {'Best_alpha':>11} {'Best_R@1':>9} {'Global_R@1':>11} {'Delta':>7}")
print('  ' + '-' * 60)
for row in alpha_scan:
    print(f"  {row['category']:<16} {row['best_alpha']:>11.2f} {row['best_r1']:>9.4f} {row['global_r1']:>11.4f} {row['delta']:>+7.4f}")

print(f"\n  Oracle system R@1: {percat['R@1']:.4f}")
print(f"  Champion R@1:      {champ['R@1']:.4f}")
print(f"  Oracle gain:       {percat['R@1'] - champ['R@1']:+.4f} (+{(percat['R@1']-champ['R@1'])*100:.2f}pp)")

In [ ]:
# Visualize per-category alphas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cats = [r['category'] for r in alpha_scan]
opt_a = [r['best_alpha'] for r in alpha_scan]
deltas = [r['delta'] for r in alpha_scan]

# Alpha values
colors = ['#2ecc71' if a != 0.4 else '#bdc3c7' for a in opt_a]
ax1.bar(cats, opt_a, color=colors, alpha=0.85)
ax1.axhline(0.4, color='blue', linestyle='--', label='Global alpha=0.4')
for i, (c, a) in enumerate(zip(cats, opt_a)):
    ax1.text(i, a+0.02, f'{a:.2f}', ha='center', fontsize=9, fontweight='bold')
ax1.set_title('Per-Category Optimal Alpha\n(Green = differs from global 0.4)')
ax1.set_ylabel('Alpha (CLIP weight)'); ax1.set_ylim(0, 1.2)
ax1.set_xticklabels(cats, rotation=25, ha='right'); ax1.legend(); ax1.grid(axis='y', alpha=0.4)

# Delta R@1
colors2 = ['#2ecc71' if d > 0.005 else '#bdc3c7' for d in deltas]
ax2.barh(cats, deltas, color=colors2, alpha=0.85)
ax2.axvline(0, color='black', linewidth=0.8)
for i, d in enumerate(deltas):
    ax2.text(max(d, 0)+0.005, i, f'{d:+.3f}', va='center', fontsize=9)
ax2.set_title('R@1 Gain from Per-Category Alpha\n(oracle, tuned on eval set)')
ax2.set_xlabel('Delta R@1'); ax2.grid(axis='x', alpha=0.4)

plt.tight_layout()
plt.show()
print(f'Key insight: 5 of 9 categories benefit from per-cat tuning')
print(f'Suiting: alpha=0.0 (pure color) — but n=2 gallery items so color alone identifies product')
print(f'Tees: alpha=0.5 — color matters more (vast majority are plain tees, differentiated by color)')

## Experiment 4.M.3: 96D Color Features (16 bins/channel)
**Hypothesis:** More color resolution → better discriminability → better retrieval.
- 48D: 8 bins/channel (RGB 24D + HSV 24D)  
- 96D: 16 bins/channel (RGB 48D + HSV 48D)

**Expected:** Small improvement from finer quantization.
**Actual: MASSIVE DROP (-23.2pp R@1). COUNTERINTUITIVE.**

In [ ]:
m96 = p4['experiments']['4M3_color96']
print('4.M.3 96D Color Features:')
print(f"  R@1: {m96['R@1']:.4f} (vs champion 0.6826)")
print(f"  Delta: {m96['delta_vs_champion']:+.4f} ({m96['delta_vs_champion']*100:+.1f}pp)")
print(f"\n  VERDICT: {m96['verdict']}")

print('\nWhy does 96D HURT so badly?')
print('  With 16 bins/channel, a navy blue product might fall in bin 11 for gallery image')
print('  but bin 10 or 12 for the query image (different lighting, viewpoint).')
print('  Histogram becomes SPARSE — most bins are zero — so cosine similarity becomes')
print('  highly sensitive to exact bin assignment.')
print('  With 8 bins, the navy blue pixels reliably land in bin 5-6 regardless of lighting.')
print('  COARSER QUANTIZATION = MORE ROBUST TO INTRA-CLASS VARIATION.')
print('  This is exactly what Liu et al. 2016 DeepFashion warned: lighting variation is')
print('  the primary intra-class challenge. Our color features must be robust to it.')

## Experiment 4.M.4: Error Analysis
**The core question:** What makes the 326 failures fail? Are they:
- **Close misses** (correct product ranked 2nd-5th — tuning can fix)
- **Hard cases** (correct product ranked 10th+ — need better features)
- **Category errors** (wrong category label — fundamental data issue)

In [ ]:
ea = p4['error_analysis']
print(f"Total queries: 1027 | Successes: {ea['n_success']} | Failures: {ea['n_fail']}")
print(f"R@1: {ea['success_rate']:.4f}\n")

print('FAILURE RANK DISTRIBUTION:')
for rank_label, stats in ea['rank_distribution'].items():
    print(f"  {rank_label.replace('_', ' '):<15}: {stats['count']:4d} failures ({stats['pct']:.1f}%)")

print(f"\nCLOSE MISSES (correct in top-5): {ea['close_miss_pct']:.1f}%")
print(f"HARD CASES (correct below rank 10): {ea['pct_fail_rank_gt10']:.1f}%")
print(f"Wrong category label: {ea['pct_wrong_category']:.1f}%")

print(f"\nSCORE GAP (top-1 wrong minus correct score):")
print(f"  Median: {ea['score_gap_median']:.4f}")
print(f"  Mean:   {ea['score_gap_mean']:.4f}")
print(f"  Tiny (<0.01): {ea['pct_tiny_gap']:.1f}% of failures")
print(f"  Small (<0.05): {ea['pct_small_gap']:.1f}% of failures")
print(f"  Large (>0.10): {ea['pct_large_gap']:.1f}% of failures")

print(f"\nSCORE DISTRIBUTION:")
print(f"  Success queries mean score: {ea['score_success_mean']:.4f}")
print(f"  Failure queries mean score: {ea['score_fail_mean']:.4f}")
print(f"  Separation: {ea['score_separation']:.4f}")

In [ ]:
# Per-category failure analysis
print('PER-CATEGORY FAILURE ANALYSIS:')
print(f"{'Category':<16} {'Fail_rate':>10} {'Med_rank':>9} {'Avg_gap':>8}")
print('-' * 50)
cat_stats = ea['per_category']
for cat, stats in sorted(cat_stats.items(), key=lambda x: -x[1]['fail_rate']):
    print(f"{cat:<16} {stats['fail_rate']:>10.4f} {stats['med_rank_correct']:>9.1f} {stats['avg_score_gap']:>8.4f}")

print('\nKEY INSIGHT:')
print('  Shorts: 50.6% failure rate — most ambiguous product type')
print('  Sweaters: 9.5% failure rate — most distinctive silhouette')
print('  Suiting: 0% failure rate — 2-item gallery = trivial retrieval')
print('  Median rank of correct product in failures: 3-5')
print('  => The bottleneck is RANKING within top-5, not RECALL')
print('  Phase 5 strategy: add a cross-modal reranker (text metadata) for top-5 re-ranking')

## Experiment 4.M.5: Multiplicative Fusion (Soft-AND)
**Hypothesis:** Additive blend `s = α*s_clip + (1-α)*s_color` allows one strong signal to dominate.
Multiplicative fusion `s = s_clip * s_color^β` requires BOTH signals to agree — soft AND.

If failures happen when CLIP and color disagree (CLIP says product A, color says product B),
multiplicative would better penalize disagreements.

In [ ]:
mult = p4['experiments']['4M5_multiplicative']
best_beta = p4['experiments']['4M5_best_beta']
delta_mult = p4['experiments']['4M5_delta_vs_additive']

print('4.M.5 Multiplicative Fusion Results:')
print(f"  {'Beta':>6}  {'R@1':>8}")
print('  ' + '-' * 20)
for key, metrics in sorted(mult.items(), key=lambda x: float(x[0].split('_')[1])):
    beta = float(key.split('_')[1])
    marker = ' <-- best' if beta == best_beta else ''
    print(f"  {beta:>6.2f}  {metrics['R@1']:>8.4f}{marker}")

print(f"\n  Best multiplicative (beta={best_beta}): R@1={mult[f'beta_{best_beta}']['R@1']:.4f}")
print(f"  Additive champion (alpha=0.4):      R@1={champ['R@1']:.4f}")
print(f"  Delta: {delta_mult:+.4f} ({delta_mult*100:+.2f}pp)")
print('\n  VERDICT: Multiplicative marginally wins at beta=1.5 (+0.29pp)')
print('  But this is ~3 queries on 1027 — statistically negligible.')
print('  Additive is simpler and more robust across conditions.')

In [ ]:
# Show the full phase 4 summary table
all_results = [
    ('P1 ResNet50 baseline (Anthony)',        0.307),
    ('P1M ResNet50 + color rerank (Mark)',    0.405),
    ('P2M CLIP B/32 bare',                   0.480),
    ('P2M CLIP B/32 + color rerank',         0.576),
    ('P3A CLIP L/14 + text + spatial',       0.6748),
    ('P3M CLIP + cat.filter',                0.5686),
    ('P3M CLIP + cat + color48 a=0.4',       0.6826),
    ('4.M.2 Per-cat alpha oracle',           percat['R@1']),
    ('4.M.5 Multiplicative beta=1.5',        mult['beta_1.5']['R@1']),
    ('4.M.3 96D color (CATASTROPHIC)',       m96['R@1']),
]
all_results.sort(key=lambda x: -x[1])

print('MASTER LEADERBOARD — All Phases:')
print(f"{'Rank':>4}  {'Experiment':<44}  {'R@1':>8}")
print('-' * 62)
for i, (name, r1) in enumerate(all_results, 1):
    mark = ' *** PHASE 4 BEST' if name == '4.M.2 Per-cat alpha oracle' else ''
    mark = ' !!! COUNTERINTUITIVE' if 'CATASTROPHIC' in name else mark
    print(f"{i:>4}  {name:<44}  {r1:>8.4f}{mark}")

## Key Findings

### 1. HEADLINE: 96D Color Catastrophically Hurts (-23.2pp) — Coarser Color is More Robust
Doubling color resolution from 8→16 bins/channel caused a catastrophic 23pp R@1 drop.
**Why:** With 16 bins, histograms become sparser. The same navy blue product lands in
different bins across gallery and query images due to lighting variation. Cosine similarity
becomes highly sensitive to bin assignment noise. 8-bin histograms tolerate lighting variation
because pixels reliably cluster into the same coarse bin regardless of illumination.

**Implication:** Color features for fashion retrieval should be lighting-robust, not
precision-maximizing. This is consistent with Liu et al. 2016 DeepFashion findings.

### 2. Per-Category Alpha Oracle: +1.27pp R@1 (0.6826→0.6952)
Different categories genuinely need different CLIP-vs-color blends:
- **Tees** (alpha=0.50): Color is the primary differentiator (most tees have similar cut)
- **Sweaters** (alpha=0.50): Style and color both matter
- **Jackets** (alpha=0.40): CLIP structure features already optimal
- **Suiting** (alpha=0.00): 2-item gallery → pure color retrieval is sufficient

### 3. 85.3% of Failures Are Close Misses (correct product in top-5)
Median score gap between top-1 wrong and correct product: 0.021. The model is
**nearly correct** on 85% of failures. This is not a recall failure — it's a precision failure.

**Implication for Phase 5:** Don't add more modalities — add a **cross-modal reranker**
that uses Anthony's text metadata (category name, style attributes) to break ties
between the top-3 candidates.

### 4. Multiplicative Fusion (beta=1.5): +0.29pp (statistically negligible)
Soft-AND fusion marginally outperforms additive at beta=1.5 but the gain (~3 queries)
is not meaningful. Additive blend is more predictable and should remain the production choice.

## Next Phase (Phase 5) Recommendations
1. **Cross-modal reranker on top-5**: Use Anthony's text metadata (product name, attributes)
   to rerank the top-5 CLIP+color candidates. If 85% of failures have correct in top-5,
   a perfect reranker could boost R@1 from 0.695 to ~0.98 (theoretical upper bound).
2. **Per-category alpha in production**: The oracle shows +1.27pp headroom from per-cat tuning.
   Tune on a 20% validation split and apply to test queries.
3. **Ablation: what if we add text for only the 85% close misses?** Text metadata may be
   the decisive signal to break the 0.021 score gap ties.

In [ ]:
# Display final plots
from IPython.display import Image, display
print('Phase 4 Results Plot:')
display(Image('../results/phase4_mark_results.png'))
print('\nError Analysis Plot:')
display(Image('../results/phase4_mark_error_analysis.png'))